In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from os import listdir
import matplotlib.pyplot as plt
from threading import Thread
from pendulibrary.integrate import integrate_state
from pendulibrary.plotters import make_gif
from tqdm.auto import tqdm

In [ ]:
fams = [f.removesuffix(".npz") for f in listdir("../database")]

In [ ]:
def make_gif_run(
    fname,
    frac,
    append_str: str = "",
    frames: int = 150,
    size: float = 5.0,
    dpi: float = 250,
    fps: int = 30,
):
    data = np.load(f"../database/{fname}.npz")
    x0s = data["x0s"].astype(np.float64)
    periods = data["periods"].astype(np.float64)
    Lr, Mr = data["params"].astype(np.float64)
    cols = np.column_stack((x0s, periods))
    steps = np.linalg.norm(np.diff(cols, axis=0), axis=1)
    Ss = np.append(0, np.cumsum(steps))
    ind = np.argmin(np.abs(Ss - Ss[-1] * frac))
    ts, xs, fs = integrate_state(x0s[ind], periods[ind], Lr, Mr, 1e-14)
    print(f"begin {fname}")
    fname_out = f"../plots/gifs/{fname}_{append_str}"
    make_gif(xs, ts, fs, Lr, fname_out, frames, size, fps, dpi)
    print(f"\tend {fname}")

In [ ]:
plt.rcParams['lines.markersize'] ** 2

In [ ]:
procs = []
for fname in tqdm(fams):
    p = Thread(target=make_gif_run, args=(fname, 1 / 10, "a"))
    procs.append(p)

for proc in procs:
    proc.start()
for proc in procs:
    proc.join()

In [ ]:
procs = []
for fname in tqdm(fams):
    p = Thread(target=make_gif_run, args=(fname, 1 / 3, "b"))
    procs.append(p)

for proc in procs:
    proc.start()
for proc in procs:
    proc.join()

In [ ]:
procs = []
for fname in tqdm(fams):
    p = Thread(target=make_gif_run, args=(fname, 1.0, "c"))
    procs.append(p)

for proc in procs:
    proc.start()
for proc in procs:
    proc.join()

In [ ]:
procs = []
for fname in tqdm(["DDsp","DDlp", "UD","DU"]):
    p = Thread(target=make_gif_run, args=(fname, 1.0, "_fin", 100, 3., 250,20))
    procs.append(p)
for proc in procs:
    proc.start()
for proc in procs:
    proc.join()